# 02 — RVC v2 Fine-Tuning
**Neural Voice Identity Control & Deepfake Audio Analysis — Disertație 2026**

Acest notebook fine-tunează RVC v2 pentru fiecare din cele 3 voci.

**Prerequisite:** Rulează mai întâi `01_data_collection.ipynb` și asigură-te că ai clipuri în `data/{voice_id}/clips/`.

**Timp estimat:** ~1-2h per voce pe Colab T4 (cu 15-30 min audio curat).

---
> ⚡ **Asigură-te că runtime-ul este setat pe GPU:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Verificare GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠ WARNING: No GPU detected. Training will be very slow.')
    print('Go to: Runtime → Change runtime type → GPU → T4')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone RVC v2 repository
import os

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Cloning RVC repository...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print('RVC already cloned.')
    !git -C {RVC_DIR} pull

os.chdir(RVC_DIR)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# Install RVC dependencies
!pip install -q -r requirements.txt
print('RVC dependencies installed.')

In [ ]:
# Download required pre-trained models (HuBERT, RMVPE, pretrained RVC v2)
import os

os.makedirs(f'{RVC_DIR}/assets/pretrained_v2', exist_ok=True)
os.makedirs(f'{RVC_DIR}/assets/uvr5_weights', exist_ok=True)

BASE_MODELS_URL = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'

files_to_download = [
    (f'{BASE_MODELS_URL}/pretrained_v2/f0D48k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0D48k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0G48k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0G48k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0D40k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0D40k.pth'),
    (f'{BASE_MODELS_URL}/pretrained_v2/f0G40k.pth',  f'{RVC_DIR}/assets/pretrained_v2/f0G40k.pth'),
    (f'{BASE_MODELS_URL}/hubert_base.pt',             f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    (f'{BASE_MODELS_URL}/rmvpe.pt',                   f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]

for url, dest in files_to_download:
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if not os.path.exists(dest):
        print(f'Downloading: {os.path.basename(dest)}')
        !wget -q '{url}' -O '{dest}'
    else:
        print(f'Already exists: {os.path.basename(dest)}')

print('\n✓ Pre-trained models ready.')

## Configurare antrenare

**TODO:** Completează configurarea de mai jos cu ID-urile vocilor tale.

In [ ]:
import os

# ============================================================
# CONFIGURARE
# ============================================================
DRIVE_BASE = '/content/drive/MyDrive/disertatie'
CLIPS_BASE = f'{DRIVE_BASE}/data/processed'
MODELS_OUT = f'{DRIVE_BASE}/models'
os.makedirs(MODELS_OUT, exist_ok=True)

VOICES_TO_TRAIN = [
    {
        'voice_id': 'voice_1',
        'display_name': 'Călin Georgescu',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
    {
        'voice_id': 'voice_2',
        'display_name': 'Nicușor Dan',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
    {
        'voice_id': 'voice_3',
        'display_name': 'Diana Șoșoacă',
        'sample_rate': 40000,
        'epochs': 200,
        'batch_size': 8,
        'save_every_epoch': 50,
    },
]
# ============================================================

print('Training configuration:')
for v in VOICES_TO_TRAIN:
    clips_dir = f"{CLIPS_BASE}/{v['voice_id']}/clips"
    clip_count = len([f for f in os.listdir(clips_dir) if f.endswith('.wav')]) if os.path.exists(clips_dir) else 0
    print(f"  {v['display_name']}: {clip_count} clips")

In [ ]:
# Training function (wraps RVC's training pipeline)
import subprocess
import shutil
import sys
sys.path.insert(0, RVC_DIR)

def train_voice(voice_config):
    voice_id = voice_config['voice_id']
    name = voice_config['display_name']
    sr = voice_config['sample_rate']
    epochs = voice_config['epochs']
    batch_size = voice_config['batch_size']
    save_interval = voice_config['save_every_epoch']
    
    clips_dir = f"{CLIPS_BASE}/{voice_id}/clips"
    log_dir = f'{RVC_DIR}/logs/{voice_id}'
    
    print(f'\n{"="*60}')
    print(f'Training: {name} ({voice_id})')
    print(f'Epochs: {epochs}, Batch: {batch_size}')
    print(f'Input clips: {clips_dir}')
    print(f'{"="*60}')
    
    # Step 1: Preprocess training data
    print('\n[1/4] Preprocessing data...')
    subprocess.run([
        'python', f'{RVC_DIR}/trainset_preprocess_pipeline_print.py',
        clips_dir, str(sr), '2',  # 2 workers
        log_dir, 'False',
    ], check=True, cwd=RVC_DIR)
    
    # Step 2: Extract features (pitch + HuBERT embeddings)
    print('\n[2/4] Extracting features (HuBERT + RMVPE)...')
    subprocess.run([
        'python', f'{RVC_DIR}/extract_feature_print.py',
        'cuda',  # device
        '1',     # n_gpus
        '0',     # gpu rank
        '0',     # exp dir index
        log_dir,
        'v2',    # RVC version
    ], check=True, cwd=RVC_DIR)
    
    # Step 3: Extract pitch (F0)
    print('\n[3/4] Extracting F0 pitch (RMVPE)...')
    subprocess.run([
        'python', f'{RVC_DIR}/extract_f0_print.py',
        log_dir, '1', 'rmvpe',
    ], check=True, cwd=RVC_DIR)
    
    # Step 4: Train model
    print(f'\n[4/4] Training ({epochs} epochs)...')
    subprocess.run([
        'python', f'{RVC_DIR}/train_nsf_sim.py',
        '-e', voice_id,
        '-sr', f'{sr//1000}k',
        '-f0', '1',
        '-bs', str(batch_size),
        '-g', '0',
        '-te', str(epochs),
        '-se', str(save_interval),
        '-pg', f'{RVC_DIR}/assets/pretrained_v2/f0G{sr//1000}k.pth',
        '-pd', f'{RVC_DIR}/assets/pretrained_v2/f0D{sr//1000}k.pth',
        '-l', '0',
        '-c', '0',
        '-sw', '0',
        '-v', 'v2',
    ], check=True, cwd=RVC_DIR)
    
    # Copy final checkpoint to Drive
    checkpoint_src = f'{log_dir}/{voice_id}_{epochs}e.pth'
    if not os.path.exists(checkpoint_src):
        # Find the latest checkpoint
        import glob
        checkpoints = sorted(glob.glob(f'{log_dir}/*.pth'))
        checkpoint_src = checkpoints[-1] if checkpoints else None
    
    if checkpoint_src and os.path.exists(checkpoint_src):
        dest = f'{MODELS_OUT}/{voice_id}.pth'
        shutil.copy2(checkpoint_src, dest)
        print(f'\n✓ Checkpoint saved: {dest}')
    else:
        print('\n⚠ Could not find final checkpoint')

print('Training function defined.')

In [ ]:
# Build FAISS index (for retrieval-augmented inference)
import subprocess

def build_index(voice_id):
    log_dir = f'{RVC_DIR}/logs/{voice_id}'
    print(f'Building FAISS index for {voice_id}...')
    
    subprocess.run([
        'python', f'{RVC_DIR}/train_index.py',
        log_dir, 'v2',
    ], check=True, cwd=RVC_DIR)
    
    # Copy index to Drive
    import glob
    index_files = glob.glob(f'{log_dir}/*.index')
    if index_files:
        dest = f'{MODELS_OUT}/{voice_id}.index'
        import shutil
        shutil.copy2(sorted(index_files)[-1], dest)
        print(f'✓ Index saved: {dest}')

print('Index function defined.')

In [ ]:
# ============================================================
# ANTRENARE — RULEAZĂ CELULA ASTA
# Durată estimată: ~1-2h per voce pe T4
# ============================================================

# Antrenează secvențial toate vocile
# Pentru a antrena doar una, comentează celelalte din VOICES_TO_TRAIN

for voice_config in VOICES_TO_TRAIN:
    try:
        train_voice(voice_config)
        build_index(voice_config['voice_id'])
        print(f"\n✓✓ DONE: {voice_config['display_name']}")
    except Exception as e:
        print(f"\n✗ ERROR for {voice_config['voice_id']}: {e}")
        import traceback
        traceback.print_exc()

print('\n' + '='*60)
print('Fine-tuning complete. Continuă cu: 03_main_app.ipynb')
print('='*60)

In [ ]:
# Test rapid de inferență — verificare că modelul funcționează
import sys
import soundfile as sf
import numpy as np

sys.path.insert(0, RVC_DIR)

TEST_VOICE_ID = 'voice_1'  # TODO: schimbă cu vocea pe care vrei să o testezi
TEST_INPUT = None  # TODO: calea către un .wav de test (sau lasă None pentru test sintetic)

if TEST_INPUT is None:
    # Generează un semnal de test sintetic
    sr = 40000
    t = np.linspace(0, 2, 2 * sr)
    test_audio = (np.sin(2 * np.pi * 200 * t) * 0.3).astype(np.float32)
    TEST_INPUT = '/tmp/test_input.wav'
    sf.write(TEST_INPUT, test_audio, sr)
    print('Created synthetic test audio.')

model_path = f'{MODELS_OUT}/{TEST_VOICE_ID}.pth'
index_path = f'{MODELS_OUT}/{TEST_VOICE_ID}.index'

if not os.path.exists(model_path):
    print(f'Model not found: {model_path}')
    print('Fine-tuning must complete first.')
else:
    from infer.modules.vc.modules import VC
    from configs.config import Config
    
    config = Config()
    vc = VC(config)
    vc.get_vc(model_path)
    
    tgt_sr, output = vc.vc_single(
        sid=0,
        input_audio_path=TEST_INPUT,
        f0_up_key=0,
        f0_file=None,
        f0_method='rmvpe',
        file_index=index_path if os.path.exists(index_path) else '',
        index_rate=0.75,
        filter_radius=3,
        resample_sr=0,
        rms_mix_rate=0.25,
        protect=0.33,
    )
    
    sf.write('/tmp/test_output.wav', output, tgt_sr)
    print(f'\n✓ Test inference successful!')
    print(f'Output sample rate: {tgt_sr}')
    print(f'Output length: {len(output)/tgt_sr:.2f}s')
    
    from IPython.display import Audio, display
    print('Input audio:')
    display(Audio(TEST_INPUT))
    print('Converted audio:')
    display(Audio('/tmp/test_output.wav'))